In [8]:
import os

DATAPATH = "/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs"

# Map each dataset name -> the exact adjacency-list file to use.
# Paths may be absolute or just a filename (resolved against DATAPATH below).
ADJ_LISTS = {
    "mnist":         "set-cover-adj-list-mnist-euclidean.txt",
    # "fashion_mnist": "set-cover-adj-list-fashion_mnist-euclidean.txt",
    # "glove25":       "adj-list-glove25-euclidean.txt",
    # "coco_i2i":      "adj-list-coco_i2i-euclidean.txt",
}

# beam_search.py knobs
BEAM_WIDTHS  = [1, 2, 4, 8, 16]
MIN_COVERAGE = 90
STEP_SIZE    = 0.5
TESTS        = 10000

print("Files in DATAPATH:")
for f in sorted(os.listdir(DATAPATH)):
    print(" ", f)

Files in DATAPATH:
  adj-list-coco_i2i-euclidean.txt
  adj-list-coco_t2i-euclidean.txt
  adj-list-fashion_mnist-euclidean.txt
  adj-list-glove25-euclidean.txt
  adj-list-mnist-euclidean.txt
  coco_i2i-512-angular.hdf5
  fashion_mnist-784-euclidean.hdf5
  glove25-25-angular.hdf5
  mnist-784-euclidean.hdf5
  set-cover-adj-list-fashion_mnist-euclidean.txt
  set-cover-adj-list-mnist-euclidean.txt


In [9]:
import glob
import subprocess
import sys
import tempfile

def find_hdf5(name):
    """Locate the .hdf5 file for a dataset name (prefix before the first '-')."""
    matches = sorted(glob.glob(os.path.join(DATAPATH, f"{name}-*.hdf5")))
    if not matches:
        raise FileNotFoundError(f"No hdf5 for dataset '{name}' in {DATAPATH}")
    return matches[0]

def resolve(path):
    """Allow bare filenames in ADJ_LISTS, resolved against DATAPATH."""
    return path if os.path.isabs(path) else os.path.join(DATAPATH, path)

beam_search_py = os.path.join(os.path.dirname(os.path.abspath("beam_search.py")),
                              "beam_search.py")
if not os.path.exists(beam_search_py):
    beam_search_py = "beam_search.py"   # fall back to cwd

# Verification-only run: beam_search.py requires --save_path and writes a CSV there.
# Point it at a throwaway temp dir so nothing persists; the numbers we want to check
# (avg recall / nodes seen / nodes expanded, and the per-coverage avg out-degree)
# are printed to stdout by print_summary BEFORE the CSV is written.
for name, adj_list in ADJ_LISTS.items():
    hdf5     = find_hdf5(name)
    adj_list = resolve(adj_list)
    if not os.path.exists(adj_list):
        raise FileNotFoundError(f"adj-list for '{name}' not found: {adj_list}")

    with tempfile.TemporaryDirectory() as tmp:
        cmd = [
            sys.executable, beam_search_py,
            "--dataset",      hdf5,
            "--adj_list",     adj_list,
            "--save_path",    tmp,           # discarded after the run
            "--beam_widths",  *[str(b) for b in BEAM_WIDTHS],
            "--min_coverage", str(MIN_COVERAGE),
            "--step_size",    str(STEP_SIZE),
            "--tests",        str(TESTS),
        ]

        print("=" * 80)
        print(f"Verifying beam search: {name}  (no results saved)")
        print("  dataset :", hdf5)
        print("  adj_list:", adj_list)
        print("=" * 80, flush=True)

        proc = subprocess.run(cmd)          # streams output live to the cell
        if proc.returncode != 0:
            print(f"!! {name} exited with code {proc.returncode}", flush=True)
        else:
            print(f"✓ {name} verified (temp CSV discarded)", flush=True)

Verifying beam search: mnist  (no results saved)
  dataset : /Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs/mnist-784-euclidean.hdf5
  adj_list: /Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs/set-cover-adj-list-mnist-euclidean.txt
Loading {'name': 'mnist', 'filepath': '/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs/mnist-784-euclidean.hdf5', 'adj_list': '/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/Built Graphs/set-cover-adj-list-mnist-euclidean.txt'}
Building networkx graphs...


60000it [00:24, 2463.04it/s]


Avg out-degrees
--------------
100.0% navigable: 17.03
99.5% navigable: 5.15
99.0% navigable: 4.08
98.5% navigable: 3.53
98.0% navigable: 3.16
97.5% navigable: 2.89
97.0% navigable: 2.68
96.5% navigable: 2.51
96.0% navigable: 2.37
95.5% navigable: 2.25
95.0% navigable: 2.14
94.5% navigable: 2.05
94.0% navigable: 1.97
93.5% navigable: 1.89
93.0% navigable: 1.83
92.5% navigable: 1.77
92.0% navigable: 1.71
91.5% navigable: 1.66
91.0% navigable: 1.62
90.5% navigable: 1.57
--------------

Searching test queries (n=10000, beam_widths=[1, 2, 4, 8, 16])...


  2%|▊                                     | 226/10000 [00:28<20:24,  7.99it/s]
Traceback (most recent call last):
  File "/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/navigable-graphs/beam_search/beam_search.py", line 223, in <module>
    main()
    ~~~~^^
  File "/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/navigable-graphs/beam_search/beam_search.py", line 154, in main
    dfs_test = run_search(Y[test_indices], query_indices=test_indices)
  File "/Users/pratyushavi/Developer/NYU-CS/Research/Navigable Graphs/navigable-graphs/beam_search/beam_search.py", line 100, in run_search
    d_q      = cdist(qvec[np.newaxis], X, metric='sqeuclidean').ravel()
               ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/anaconda3/envs/aws/lib/python3.13/site-packages/scipy/spatial/distance.py", line 2926, in cdist
    return cdist_fn(XA, XB, out=out, **kwargs)
KeyboardInterrupt


KeyboardInterrupt: 